In [1]:
import os
import sys
import numpy as np
sys.path.append("../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune_head import (
    head_importance_prunning, compute_heads_importance
)

In [3]:
name = "IMDB"
device = torch.device("cuda:0")
checkpoint = None
batch_size = 16
num_workers = 4
num_samples = 128
ci_ratio = 0.4
seed = 44
include_layers = ["attention", "intermediate", "output"]
exclude_layers = None

In [4]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.
{'model_name': 'textattack/bert-base-uncased-imdb', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'IMDB', 'num_labels': 2, 'cache_dir': 'Models'}
The model textattack/bert-base-uncased-imdb is loaded.


In [5]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name, batch_size=32, num_workers=48
)

Loading cached dataset IMDB.
train.pkl is loaded from cache.
valid.pkl is loaded from cache.
test.pkl is loaded from cache.
The dataset IMDB is loaded
{'dataset_name': 'IMDB', 'path': 'imdb', 'config_name': 'plain_text', 'text_column': 'text', 'label_column': 'label', 'cache_dir': 'Datasets/IMDB', 'task_type': 'classification'}


In [6]:
train = copy.deepcopy(train_dataloader)

In [7]:
positive_samples = SamplingDataset(
    train, 0, num_samples, num_labels, True, 4, device=device, resample=False, seed=seed
)

In [8]:
attn_entropy, head_importance, preds, labels, per_class_importance_list =compute_heads_importance(
    model, model_config, positive_samples
)

In [9]:
print(head_importance)

[[7.45387435e-01 1.31172344e-01 2.09457934e-01 4.53669012e-01
  6.55563891e-01 1.70871750e-01 6.24198973e-01 1.86340407e-01
  3.75580490e-01 3.05276781e-01 3.51678461e-01 8.04741904e-02]
 [1.53663844e-01 4.75446075e-01 9.93110836e-02 5.56330793e-02
  4.79663163e-01 2.22783625e-01 5.17108440e-01 5.40231466e-01
  3.19783628e-01 2.64881045e-01 6.94132864e-01 5.44894814e-01]
 [5.83898067e-01 5.21048903e-01 1.85945094e-01 2.12308496e-01
  1.41157344e-01 7.46986985e-01 4.47151035e-01 2.59935558e-01
  1.76380485e-01 6.13723636e-01 2.10632786e-01 2.56455272e-01]
 [4.11985606e-01 5.36107600e-01 1.44055262e-01 6.04393065e-01
  4.30638678e-02 5.47299802e-01 1.36026844e-01 5.07688224e-01
  2.72482574e-01 4.18429703e-01 2.98314005e-01 5.25472403e-01]
 [2.64497489e-01 3.80934566e-01 4.19298857e-01 4.51580435e-01
  2.66124278e-01 5.30367732e-01 3.08098584e-01 3.35738242e-01
  2.71233976e-01 5.86898744e-01 5.52983224e-01 3.20082873e-01]
 [3.44000220e-01 6.44266233e-02 1.12959154e-01 4.37563777e-01
  7

In [10]:
import math
import torch.nn as nn

self_outputs = []
forward_hooks = []

def forward_hook(module, input, output):
    hidden_states = input[0]
    attention_mask = input[1]
    mixed_query_layer = module.query(hidden_states)
    mixed_key_layer = module.key(hidden_states)
    mixed_value_layer = module.value(hidden_states)
    
    query_layer = module.transpose_for_scores(mixed_query_layer)
    key_layer = module.transpose_for_scores(mixed_key_layer)
    value_layer = module.transpose_for_scores(mixed_value_layer)

    attention_scores = torch.matmul(
        query_layer, key_layer.transpose(-1, -2))
    attention_scores = attention_scores / math.sqrt(module.attention_head_size)

    if attention_mask is not None:
        attention_scores = attention_scores + attention_mask

    attention_probs = nn.Softmax(dim=-1)(attention_scores)
    attention_probs = module.dropout(attention_probs)

    context_layer = torch.matmul(attention_probs, value_layer)
    
    context_layer2 = context_layer.permute(0, 2, 1, 3).contiguous()
    
    new_context_layer_shape = context_layer2.size()[:-2] + (module.all_head_size,)
    context_layer3 = context_layer2.view(*new_context_layer_shape)
    
    output[0].requires_grad_(True)
    output[0].retain_grad()
    self_outputs.append({
        "context_layer_val": context_layer,
        "context_layer": output[0],
        "shape1": context_layer.shape,
        "shape2": context_layer2.shape,
        "shape3": context_layer3.shape
    })
    return output
    
def register_hooks(model):
    for layer in range(model.bert.config.num_hidden_layers):
        self_att = model.bert.encoder.layer[layer].attention.self
        self_att.register_forward_hook(forward_hook)

def calculate_head_importance(
        model,
        dataloader,
        device=None,
        normalize_scores_by_layer=True,
        subset_size=1.0,
):
    """Calculate head importance scores"""
    register_hooks(model)
    
    # Disable dropout
    model.eval()
    # Device
    device = device or next(model.parameters()).device
    # if subset_size <= 1:
    #     subset_size *= len(data)
    n_prune_steps = int(np.ceil(int(subset_size) / batch_size))

    # Head importance tensor
    n_layers = model.bert.config.num_hidden_layers
    n_heads = model.bert.config.num_attention_heads
    head_importance = torch.zeros(n_layers, n_heads).to(device)
    tot_tokens = 0

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        input_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        model.zero_grad()

        # Compute gradients
        output = model(input_ids, labels=labels, attention_mask=input_mask)
        loss = output.loss
        loss.backward()
        
        for layer in range(model.bert.config.num_hidden_layers):
            ctx = self_outputs[layer]["context_layer"]
            ctx_val = self_outputs[layer]["context_layer_val"]
            shape1 = self_outputs[layer]["shape1"]
            shape2 = self_outputs[layer]["shape2"]
            shape3 = self_outputs[layer]["shape3"]
            
            grads = ctx.grad.view(shape2)
            grads = grads.permute(0, 2, 1, 3)

            dot = torch.einsum("bhli,bhli->bhl", [grads, ctx_val])
            head_importance[layer] += dot.abs().sum(-1).sum(0).detach()

            del grads, ctx, ctx_val, dot

        tot_tokens += input_mask.float().detach().sum().data

In [11]:
hi1 = calculate_head_importance(
    model, 
    train_dataloader, 
    device="cuda", 
    normalize_scores_by_layer=True,
    subset_size=1.0,
)
for hook in forward_hooks:
    hook.remove()

OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 23.62 GiB of which 34.88 MiB is free. Including non-PyTorch memory, this process has 23.05 GiB memory in use. Of the allocated memory 22.50 GiB is allocated by PyTorch, and 97.42 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)